# Lecture 10: Editable Points and Covariance $\Sigma$

- Edit a small set of points (x1,y1,...).
- Observe Euclidean vs Mahalanobis distances from a query point.
- Control covariance with $(\sigma_x, \sigma_y, \rho)$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display


def parse_points(txt):
    pts = []
    for row in txt.strip().split(';'):
        row=row.strip()
        if not row:
            continue
        a,b = row.split(',')
        pts.append([float(a), float(b)])
    arr=np.array(pts, dtype=float)
    if arr.ndim!=2 or arr.shape[1]!=2 or len(arr)==0:
        raise ValueError('Expected format: x1,y1; x2,y2; ...')
    return arr


def draw(points_text='0,0; 1,1; 2,0.5; -1,1.6; -2,-0.8', qx=0.0, qy=0.0, sigma_x=1.0, sigma_y=1.0, rho=0.0):
    pts = parse_points(points_text)
    q = np.array([qx, qy])

    cov = np.array([[sigma_x**2, rho*sigma_x*sigma_y],
                    [rho*sigma_x*sigma_y, sigma_y**2]])
    inv = np.linalg.pinv(cov)

    dif = pts - q
    d_e = np.sqrt((dif**2).sum(axis=1))
    d_m = np.sqrt(np.sum((dif @ inv) * dif, axis=1))

    fig, ax = plt.subplots(figsize=(6.5,6))
    ax.scatter(pts[:,0], pts[:,1], c='#2563eb', s=45, label='data points')
    ax.scatter([q[0]], [q[1]], c='#dc2626', s=90, marker='x', label='query')

    # covariance ellipse
    vals, vecs = np.linalg.eigh(cov)
    t = np.linspace(0, 2*np.pi, 240)
    unit = np.vstack([np.cos(t), np.sin(t)])
    ell = vecs @ (np.sqrt(vals)[:,None] * unit)
    ax.plot(q[0] + ell[0], q[1] + ell[1], 'k--', lw=1.5, label='1-sigma ellipse')

    for i,p in enumerate(pts):
        ax.plot([q[0], p[0]], [q[1], p[1]], color='gray', alpha=0.4)
        ax.text(p[0]+0.06, p[1]+0.06, f'{i+1}: E={d_e[i]:.2f}, M={d_m[i]:.2f}', fontsize=8)

    ax.set_title('Point distances with covariance-aware metric')
    ax.grid(alpha=0.25)
    ax.legend(loc='best')
    plt.show()

    order_e = np.argsort(d_e)
    order_m = np.argsort(d_m)
    print('Nearest by Euclidean:', [int(i+1) for i in order_e[:3]])
    print('Nearest by Mahalanobis:', [int(i+1) for i in order_m[:3]])

widgets.interact(
    draw,
    points_text=widgets.Text(description='points', value='0,0; 1,1; 2,0.5; -1,1.6; -2,-0.8', layout=widgets.Layout(width='850px')),
    qx=widgets.FloatSlider(description='q_x', min=-3, max=3, step=0.1, value=0.0, continuous_update=False),
    qy=widgets.FloatSlider(description='q_y', min=-3, max=3, step=0.1, value=0.0, continuous_update=False),
    sigma_x=widgets.FloatSlider(description='sigma_x', min=0.2, max=3, step=0.1, value=1.0, continuous_update=False),
    sigma_y=widgets.FloatSlider(description='sigma_y', min=0.2, max=3, step=0.1, value=1.0, continuous_update=False),
    rho=widgets.FloatSlider(description='rho', min=-0.95, max=0.95, step=0.05, value=0.0, continuous_update=False),
)
